# AgriYield AI — Exploratory Data Analysis & Quality Audit
### District-wise, Season-wise Crop Production Statistics from 1997

**Author:** Senior Machine Learning Engineer & Agricultural Data Scientist  
**Dataset Publisher:** Directorate of Economics and Statistics (DES), Ministry of Agriculture and Farmers Welfare, Government of India  
**Portal:** [data.gov.in](https://data.gov.in/resource/district-wise-season-wise-crop-production-statistics-1997)  
**License:** Government Open Data License - India (GODL)

## 1. Project Background & Objective
The objective of this exploratory data analysis is to thoroughly investigate the authentic Government of India crop production dataset covering 1997 through 2015. We will:
1. Validate the raw schema, column data types, and record volume.
2. Audit missing values, zero observations, and duplicate keys.
3. Uncover critical domain anomalies (e.g. Coconut reported in *nuts* rather than metric tonnes).
4. Compute and audit agricultural yield ($\text{Yield} = \frac{\text{Production}}{\text{Area}}$ in Tonnes/Hectare).
5. Analyze long-term temporal trends, regional disparities, and seasonal variations.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.load_data import load_raw_data, validate_raw_schema
from src.data.preprocess import clean_text_columns, audit_and_clean_data

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.figsize'] = (10, 5)
print('Environment initialized successfully.')

## 2. Ingesting Raw Official Dataset & Schema Verification

In [ ]:
raw_df = load_raw_data()
print(f'Raw Dataset Dimensions: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns')
is_valid, missing = validate_raw_schema(raw_df)
print(f'Schema Validation: {"PASSED" if is_valid else "FAILED"}')
raw_df.head()

## 3. Data Quality, Missing Values & Duplicate Inspection

In [ ]:
null_summary = pd.DataFrame({
    'Data_Type': raw_df.dtypes,
    'Null_Count': raw_df.isnull().sum(),
    'Null_Percentage': (raw_df.isnull().sum() / len(raw_df) * 100).round(3)
})
print('Missing Values Audit:')
print(null_summary)

# Analytical key duplicate check
key_cols = ['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop']
print(f'\nExact Duplicate Rows: {raw_df.duplicated().sum()}')
print(f'Duplicate Records on Key {key_cols}: {raw_df.duplicated(subset=key_cols).sum()}')

### Finding: Missing Production Treatment
Exactly **3,730 records (1.516%)** have null values in `Production`. Sown `Area` has 0 missing values.
Because `Production` is the outcome variable from which `Yield` is calculated, imputing production would create synthetic ground truth labels. We document this and exclude these rows from supervised yield modeling.

## 4. Uncovering the Unit Anomaly: The Coconut Discrepancy

In [ ]:
cleaned_df = clean_text_columns(raw_df.dropna(subset=['Production']))
cleaned_df['Yield'] = cleaned_df['Production'] / cleaned_df['Area']

top_crops_by_yield = cleaned_df.groupby('Crop')['Yield'].agg(['median', 'mean', 'max']).sort_values('median', ascending=False)
print('Top 10 Crops by Median Yield:')
print(top_crops_by_yield.head(10))

coconut_data = cleaned_df[cleaned_df['Crop'].str.contains('Coconut', case=False)]
print(f'\nCoconut Maximum Yield: {coconut_data["Yield"].max():,.1f} units/ha | Mean: {coconut_data["Yield"].mean():,.1f}')

> **Agronomic Reality:** In official Indian agricultural reporting (DES), standard field crops (Rice, Wheat, Pulses) are recorded in **Metric Tonnes**, but **Coconut** is recorded in **nuts / thousand nuts**. Treating coconut nuts as metric tonnes would distort regression loss functions. Our pipeline assigns an explicit `Unit_Type` flag to maintain mathematical rigor.

## 5. Multi-Year Crop Yield Trends

In [ ]:
major_crops = ['Rice', 'Wheat', 'Maize', 'Sugarcane', 'Groundnut']
trend_data = cleaned_df[cleaned_df['Crop'].isin(major_crops)].groupby(['Crop_Year', 'Crop'])['Yield'].mean().unstack()

fig, ax = plt.subplots(figsize=(11, 5))
for crop in major_crops:
    if crop in trend_data.columns and crop != 'Sugarcane': # Sugarcane is on higher biomass scale
        ax.plot(trend_data.index, trend_data[crop], marker='o', label=crop, linewidth=2)
ax.set_title('National Average Yield Trends for Food Grains (1997–2015)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Crop Year', fontsize=12)
ax.set_ylabel('Mean Yield (Metric Tonnes / Hectare)', fontsize=12)
ax.legend(title='Crop', frameon=True)
plt.tight_layout()
plt.show()

## 6. Seasonality & Regional Variations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
season_order = ['Kharif', 'Rabi', 'Summer', 'Autumn', 'Winter', 'Whole Year']
rice_data = cleaned_df[(cleaned_df['Crop'] == 'Rice') & (cleaned_df['Yield'] <= 8.0)]
sns.boxplot(data=rice_data, x='Season', y='Yield', order=season_order, palette='Greens_r', ax=ax)
ax.set_title('Rice Yield Variation by Cropping Season', fontsize=13, fontweight='bold', pad=10)
ax.set_ylabel('Yield (Tonnes / Ha)', fontsize=11)
plt.tight_layout()
plt.show()

## 7. Conclusions & Modeling Readiness
1. **Clean Records**: 242,361 valid observations retained.
2. **Units Standardized**: Food grains and commercial crops evaluated in metric tonnes per hectare; non-tonne crops handled separately.
3. **Target Variable**: $\text{Yield} = \frac{\text{Production}}{\text{Area}}$ (Tonnes/Ha).
4. **No Leakage Protocol**: Sown area (`Area`) is known at planting; post-harvest `Production` is strictly excluded from feature sets.